<a href="https://colab.research.google.com/github/Jose-Bautista-gnss/Evaluacion_Grupal_1/blob/main/codigo/evaluacion_grupal_1_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab"/></a>

# Homicidios en los departamentos de Colombia durante 2025

## Evaluación grupal 1

**Integrantes:**

- Bautista Janampa José — 20220430
- Saldaña Ruiz María Milagros — 20241445
- Díaz Sulca Andrea Ariana — 20245499

En este trabajo se analiza la distribución de los homicidios registrados en los 32 departamentos de Colombia durante 2025. Bogotá D. C. no se incluye porque es un Distrito Capital y no un departamento. La variable principal es `homicidios_2025` y la población proyectada para el mismo año se utiliza como referencia para construir una tasa comparable entre departamentos.

Se elaboran los tres mapas temáticos solicitados: mapa de densidad de puntos (DDM), mapa de símbolos proporcionales (PSM) y mapa coroplético. Los dos primeros representan cantidades absolutas. La coropleta responde una pregunta distinta, pues muestra los homicidios por cada 100 000 habitantes.

El CSV de homicidios fue preparado sumando las categorías de la fuente policial cuyo nombre comienza con “Homicidio”. Por ello incluye homicidio intencional y homicidio por tránsito vehicular. Esta definición debe conservarse al interpretar los resultados.

## Preparación del entorno

El cuaderno sigue la estructura de la evaluación individual y puede ejecutarse tanto en VS Code como en Google Colab. La primera celda instala únicamente las bibliotecas que podrían faltar. Después se importan las herramientas utilizadas en la lectura, revisión y representación de los datos.

In [ ]:
%%capture
%pip install -q geopandas mapclassify pyogrio

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
import re
import unicodedata

import geopandas as gpd
import mapclassify
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

## Archivos de trabajo

Los tres archivos se leen desde el repositorio de GitHub. De esta forma, el análisis no depende de una ruta local como `D:/...` y puede reproducirse en otra computadora. Los CSV se leen directamente y el GeoPackage se descarga temporalmente porque este formato necesita acceso binario para consultar sus capas.

In [ ]:
repo_raw = (
    "https://raw.githubusercontent.com/"
    "Jose-Bautista-gnss/Evaluacion_Grupal_1/main/data/"
)

gpkg_url = repo_raw + "departamentos_colombia.gpkg"
homicidios_url = repo_raw + "homicidios_2025.csv"
poblacion_url = repo_raw + "poblacion_2025.csv"

temporary_directory = TemporaryDirectory()
gpkg_path = Path(temporary_directory.name) / "departamentos_colombia.gpkg"
urlretrieve(gpkg_url, gpkg_path)

print("GeoPackage descargado desde GitHub.")

Antes de leer las geometrías se revisan las capas del GeoPackage. El código prioriza una capa llamada `departamentos`; si el archivo utiliza otro nombre, identifica automáticamente la única capa poligonal disponible.

In [ ]:
layers = gpd.list_layers(gpkg_path)
layers

In [ ]:
polygon_layers = layers[
    layers["geometry_type"].str.contains("Polygon", case=False, na=False)
].copy()

if "departamentos" in layers["name"].tolist():
    layer_name = "departamentos"
elif len(polygon_layers) == 1:
    layer_name = polygon_layers.iloc[0]["name"]
else:
    raise ValueError(
        "No se pudo identificar de manera inequívoca la capa departamental. "
        f"Capas disponibles: {layers['name'].tolist()}"
    )

departments_raw = gpd.read_file(gpkg_path, layer=layer_name)

print("Capa utilizada:", layer_name)

departments_raw.head()

También se revisan el CRS y los campos disponibles. El código busca primero nombres de columnas departamentales frecuentes; si ninguno coincide, selecciona una columna de texto que contenga exactamente 32 nombres diferentes.

In [ ]:
preferred_name_columns = [
    "DEP_NOMBRE", "departamento", "Departamento", "DEPARTAMENTO",
    "DPTO_CNMBR", "NOMBRE_DPT", "NOM_DEP", "NAME_1", "shapeName"
]

name_column = next(
    (column for column in preferred_name_columns if column in departments_raw.columns),
    None
)

if name_column is None:
    text_candidates = [
        column for column in departments_raw.select_dtypes(
            include=["object", "string"]
        ).columns
        if departments_raw[column].nunique(dropna=True) == 32
    ]
    if len(text_candidates) != 1:
        raise ValueError(
            "No se pudo identificar el campo con los nombres departamentales. "
            f"Candidatos encontrados: {text_candidates}"
        )
    name_column = text_candidates[0]

departments_raw.info()
print("CRS:", departments_raw.crs)
print("Campo de nombres:", name_column)
print("Geometrías originales:", len(departments_raw))
print("Nombres diferentes:", departments_raw[name_column].nunique(dropna=True))
print("Registros sin nombre:", departments_raw[name_column].isna().sum())

Los CSV reúnen los indicadores ya procesados por departamento. Se leen sus primeras filas y se comprueba que tengan las columnas necesarias.

In [ ]:
homicides = pd.read_csv(homicidios_url, encoding="utf-8-sig")
population = pd.read_csv(poblacion_url, encoding="utf-8-sig")

display(homicides.head())
display(population.head())

In [ ]:
required_homicides = {"departamento", "homicidios_2025"}
required_population = {"departamento", "poblacion_2025"}

assert required_homicides.issubset(homicides.columns), (
    f"Faltan columnas en homicidios: {required_homicides - set(homicides.columns)}"
)
assert required_population.issubset(population.columns), (
    f"Faltan columnas en población: {required_population - set(population.columns)}"
)

homicides["homicidios_2025"] = pd.to_numeric(
    homicides["homicidios_2025"], errors="raise"
)
population["poblacion_2025"] = pd.to_numeric(
    population["poblacion_2025"], errors="raise"
)

assert len(homicides) == 32, (
    f"Se esperaban 32 departamentos en homicidios, pero hay {len(homicides)}."
)
assert len(population) == 32, (
    f"Se esperaban 32 departamentos en población, pero hay {len(population)}."
)

data_review = pd.DataFrame({
    "tabla": ["Homicidios", "Población"],
    "filas": [len(homicides), len(population)],
    "duplicados": [
        homicides["departamento"].duplicated().sum(),
        population["departamento"].duplicated().sum()
    ],
    "valores_faltantes": [
        homicides[list(required_homicides)].isna().sum().sum(),
        population[list(required_population)].isna().sum().sum()
    ]
})

print("Homicidios registrados:", f"{homicides['homicidios_2025'].sum():,.0f}")
print("Población proyectada:", f"{population['poblacion_2025'].sum():,.0f}")
data_review

## Preparación de los nombres y las geometrías

Los tres archivos escriben algunos departamentos de manera diferente. Por ejemplo, el mapa usa `NORTESAN`, mientras que los CSV utilizan `Norte de Santander`; también aparecen diferencias en tildes y mayúsculas. Se crea una llave normalizada para unir sin modificar los nombres originales.

La normalización no utiliza coincidencia difusa automática. En un conjunto de solo 32 unidades es más seguro revisar y corregir explícitamente los pocos casos conocidos.

In [ ]:
def normalize_department(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^A-Z0-9 ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


shape_changes = {
    "NORTESAN": "NORTE DE SANTANDER",
    "GUAJIRA": "LA GUAJIRA",
    "VALLE": "VALLE DEL CAUCA",
    "SAN ANDRES": "SAN ANDRES Y PROVIDENCIA",
    "SAN ANDRES PROVIDENCIA Y SANTA CATALINA": "SAN ANDRES Y PROVIDENCIA",
    "ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA": "SAN ANDRES Y PROVIDENCIA",
    "ARCHIPIELAGO DE SAN ANDRES Y PROVIDENCIA": "SAN ANDRES Y PROVIDENCIA"
}

csv_changes = {
    "GUAJIRA": "LA GUAJIRA",
    "VALLE": "VALLE DEL CAUCA",
    "SAN ANDRES": "SAN ANDRES Y PROVIDENCIA"
}

departments_raw["departamento_clave"] = (
    departments_raw[name_column]
    .map(normalize_department)
    .replace(shape_changes)
)
homicides["departamento_clave"] = (
    homicides["departamento"]
    .map(normalize_department)
    .replace(csv_changes)
)
population["departamento_clave"] = (
    population["departamento"]
    .map(normalize_department)
)

expected_departments = {
    "AMAZONAS", "ANTIOQUIA", "ARAUCA", "ATLANTICO", "BOLIVAR",
    "BOYACA", "CALDAS", "CAQUETA", "CASANARE", "CAUCA", "CESAR",
    "CHOCO", "CORDOBA", "CUNDINAMARCA", "GUAINIA", "GUAVIARE",
    "HUILA", "LA GUAJIRA", "MAGDALENA", "META", "NARINO",
    "NORTE DE SANTANDER", "PUTUMAYO", "QUINDIO", "RISARALDA",
    "SAN ANDRES Y PROVIDENCIA", "SANTANDER", "SUCRE", "TOLIMA",
    "VALLE DEL CAUCA", "VAUPES", "VICHADA"
}

assert set(homicides["departamento_clave"]) == expected_departments, (
    "Los departamentos del CSV de homicidios no coinciden con los 32 esperados."
)
assert set(population["departamento_clave"]) == expected_departments, (
    "Los departamentos del CSV de población no coinciden con los 32 esperados."
)

departments_raw[[name_column, "departamento_clave"]].drop_duplicates().head(10)

Se eliminan únicamente los registros sin nombre o sin geometría, se reparan las geometrías y se aplica `dissolve()` para garantizar una sola geometría multipartes por departamento. Este procedimiento funciona tanto si el archivo ya contiene 32 registros como si algunos departamentos están divididos en varios fragmentos.

In [ ]:
departments_clean = departments_raw.dropna(
    subset=["departamento_clave"]
).copy()

departments_clean = departments_clean[
    ~departments_clean.geometry.is_empty
].copy()
departments_clean["geometry"] = departments_clean.geometry.make_valid()

departments = departments_clean.dissolve(
    by="departamento_clave",
    as_index=False
)[["departamento_clave", "geometry"]]

print("Geometrías después de agrupar:", len(departments))
assert len(departments) == 32, (
    f"Se esperaban 32 geometrías departamentales, pero hay {len(departments)}."
)
assert set(departments["departamento_clave"]) == expected_departments, (
    "Los nombres del GeoPackage no coinciden con los 32 departamentos esperados."
)
departments.head()

Primero se unen las dos tablas de indicadores. La validación `one_to_one` comprueba que cada departamento aparezca una sola vez en ambos CSV.

In [ ]:
indicators = population[
    ["departamento_clave", "departamento", "poblacion_2025"]
].merge(
    homicides[["departamento_clave", "homicidios_2025"]],
    on="departamento_clave",
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert (indicators["_merge"] == "both").all(), (
    "Existen departamentos que no coinciden entre los dos CSV."
)
indicators.drop(columns="_merge", inplace=True)

indicators.info()

## Revisión de la unión espacial

Antes de crear los mapas se comparan las llaves. Esta revisión evita que una unidad territorial desaparezca silenciosamente durante el `merge`.

In [ ]:
only_data = sorted(
    set(indicators["departamento_clave"])
    - set(departments["departamento_clave"])
)
only_map = sorted(
    set(departments["departamento_clave"])
    - set(indicators["departamento_clave"])
)

print("Con datos pero sin geometría:", only_data)
print("Con geometría pero sin datos:", only_map)

assert only_data == [], (
    f"Departamentos con datos pero sin geometría: {only_data}"
)
assert only_map == [], (
    f"Departamentos con geometría pero sin datos: {only_map}"
)

La auditoría exige una correspondencia completa entre los tres archivos. El análisis solo continúa cuando los 32 departamentos tienen geometría, población y registros de homicidios; así se evita que una unidad desaparezca silenciosamente de los mapas.

In [ ]:
map_and_data = departments.merge(
    indicators,
    on="departamento_clave",
    how="left",
    validate="one_to_one"
)

merge_review = pd.Series({
    "unidades_en_el_mapa": len(map_and_data),
    "homicidios_faltantes": map_and_data["homicidios_2025"].isna().sum(),
    "poblacion_faltante": map_and_data["poblacion_2025"].isna().sum(),
    "geometrias_invalidas": (~map_and_data.geometry.is_valid).sum(),
    "geometrias_vacias": map_and_data.geometry.is_empty.sum()
})

assert map_and_data[["homicidios_2025", "poblacion_2025"]].notna().all().all()
merge_review

Para los tres mapas se utiliza `EPSG:9377`, correspondiente a MAGNA-SIRGAS 2018 / Origen-Nacional. Trabajar en metros evita hacer el muestreo de puntos y la ubicación de símbolos directamente sobre coordenadas angulares.

In [ ]:
colombia_9377 = map_and_data.to_crs(9377)

minx, miny, maxx, maxy = colombia_9377.total_bounds
x_margin = (maxx - minx) * 0.04
y_margin = (maxy - miny) * 0.04
x_limits = (minx - x_margin, maxx + x_margin)
y_limits = (miny - y_margin, maxy + y_margin)

colombia_9377.crs

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
colombia_9377.plot(
    ax=ax,
    color="#d9eaf7",
    edgecolor="grey",
    linewidth=0.5
)
ax.set_title("Departamentos disponibles para el análisis")
ax.set_xlim(*x_limits)
ax.set_ylim(*y_limits)
ax.set_axis_off()
plt.show()

# 1. Mapa de densidad de puntos

En un DDM todos los puntos tienen el mismo tamaño y el mismo valor. Aquí cada punto representa aproximadamente cinco homicidios registrados. Cuando el valor no es múltiplo de cinco, el número de puntos se redondea y se garantiza al menos uno para no hacer desaparecer a los departamentos con valores pequeños.

Los puntos se distribuyen aleatoriamente dentro de cada polígono. No indican el lugar exacto donde ocurrió un hecho; solo representan la cantidad departamental.

In [ ]:
dot_map = colombia_9377.copy()
dot_value = 5

dot_map["num_dots"] = np.maximum(
    1,
    np.rint(dot_map["homicidios_2025"] / dot_value).astype(int)
)

dot_map[
    ["departamento", "homicidios_2025", "num_dots"]
].sort_values("homicidios_2025", ascending=False).head(10)

In [ ]:
sampled_points = dot_map.geometry.sample_points(
    size=dot_map["num_dots"],
    rng=12345
)

homicide_dots = gpd.GeoDataFrame(
    geometry=sampled_points.explode(index_parts=False).reset_index(drop=True),
    crs=colombia_9377.crs
)

print("Puntos representados:", len(homicide_dots))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))

colombia_9377.plot(
    ax=ax,
    facecolor="white",
    edgecolor="grey",
    linewidth=0.5
)
homicide_dots.plot(
    ax=ax,
    markersize=1.2,
    color="#cb181d"
)

dot_legend = Line2D(
    [0], [0],
    marker="o",
    color="none",
    markerfacecolor="#cb181d",
    markeredgecolor="none",
    markersize=5,
    label="1 punto ≈ 5 homicidios"
)

ax.legend(handles=[dot_legend], loc="lower left", frameon=False)
ax.set_title("Homicidios registrados en Colombia durante 2025 - DDM")
ax.set_xlim(*x_limits)
ax.set_ylim(*y_limits)
ax.set_axis_off()
plt.show()

El DDM permite reconocer dónde se concentra una mayor parte del total registrado. Los departamentos con más puntos dominan la imagen, pero esto todavía no significa que tengan una mayor intensidad relativa: también suelen concentrar más población.

# 2. Mapa de símbolos proporcionales

El PSM representa nuevamente los valores absolutos, pero utiliza un círculo por departamento. La posición se obtiene con `representative_point()`, que mantiene el símbolo dentro de su geometría. La raíz cuadrada comprime la diferencia visual entre valores muy altos y bajos, siguiendo el criterio utilizado en la evaluación individual.

In [ ]:
symbol_map = colombia_9377.copy()
symbol_map["geometry"] = symbol_map.representative_point()

symbol_scale = 6
symbol_map["size"] = (
    np.sqrt(symbol_map["homicidios_2025"]) * symbol_scale
)

symbol_map[
    ["departamento", "homicidios_2025", "size"]
].sort_values("homicidios_2025", ascending=False).head(10)

Antes de dibujar el PSM se identifican los valores atípicos con el criterio intercuartílico. Un valor se considera atípico superior cuando supera $Q_3 + 1.5\,IQR$. Esta revisión se calcula directamente sobre los datos y no depende del gráfico.

In [ ]:
q1 = symbol_map["homicidios_2025"].quantile(0.25)
q3 = symbol_map["homicidios_2025"].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

symbol_map["es_atipico"] = ~symbol_map["homicidios_2025"].between(
    lower_limit,
    upper_limit
)

symbol_map[
    symbol_map["es_atipico"]
][["departamento", "homicidios_2025"]].sort_values(
    "homicidios_2025",
    ascending=False
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(
    colombia_9377["homicidios_2025"],
    vert=False,
    patch_artist=True,
    boxprops={"facecolor": "#9ecae1"},
    medianprops={"color": "#08519c", "linewidth": 2}
)
ax.set_title("Revisión de valores atípicos de homicidios")
ax.set_xlabel("Homicidios registrados")
ax.set_yticks([])
plt.show()

In [ ]:
symbols_regular = symbol_map[~symbol_map["es_atipico"]].copy()
symbols_outlier = symbol_map[symbol_map["es_atipico"]].copy()

fig, ax = plt.subplots(figsize=(9, 10))

colombia_9377.plot(
    ax=ax,
    color="white",
    edgecolor="grey",
    linewidth=0.5
)
symbols_regular.plot(
    ax=ax,
    markersize=symbols_regular["size"],
    color="#6baed6",
    edgecolor="#2171b5",
    linewidth=0.5,
    alpha=0.75
)
symbols_outlier.plot(
    ax=ax,
    markersize=symbols_outlier["size"],
    color="#fdae6b",
    edgecolor="#e6550d",
    linewidth=0.6,
    alpha=0.80
)

size_values = [100, 1_000, 3_000]
size_handles = [
    ax.scatter(
        [], [],
        s=np.sqrt(value) * symbol_scale,
        facecolor="#6baed6",
        edgecolor="#2171b5",
        alpha=0.75,
        label=f"{value:,}".replace(",", " ")
    )
    for value in size_values
]

size_legend = ax.legend(
    handles=size_handles,
    title="Homicidios",
    loc="lower left",
    frameon=False
)
ax.add_artist(size_legend)

color_handles = [
    Patch(facecolor="#6baed6", edgecolor="#2171b5", label="Valor no atípico"),
    Patch(facecolor="#fdae6b", edgecolor="#e6550d", label="Valor atípico")
]
ax.legend(
    handles=color_handles,
    title="Clasificación",
    loc="upper right",
    frameon=False
)

ax.set_title("Homicidios registrados en Colombia durante 2025 - PSM")
ax.set_xlim(*x_limits)
ax.set_ylim(*y_limits)
ax.set_axis_off()
plt.show()

El PSM facilita la comparación entre departamentos porque cada unidad tiene un solo símbolo. Los círculos naranjas indican valores que se apartan claramente del conjunto. Igual que el DDM, este mapa presenta cantidades totales y todavía está influido por el tamaño de la población.

# 3. Mapa coroplético

Una coropleta con el número bruto de homicidios podría sugerir mayor intensidad únicamente porque un departamento tiene más habitantes. Por ello se calcula una tasa por cada 100 000 habitantes:

$$
\text{Tasa de homicidios}
= \frac{\text{homicidios registrados}}{\text{población proyectada}}\times 100\,000
$$

La tasa permite comparar departamentos con tamaños poblacionales diferentes. No debe interpretarse como una tasa exclusiva de violencia intencional porque el CSV también incluye homicidios por tránsito.

In [ ]:
colombia_9377["tasa_homicidios_100000"] = (
    colombia_9377["homicidios_2025"]
    / colombia_9377["poblacion_2025"]
    * 100_000
)

colombia_9377[
    [
        "departamento",
        "homicidios_2025",
        "poblacion_2025",
        "tasa_homicidios_100000"
    ]
].sort_values(
    "tasa_homicidios_100000",
    ascending=False
).head(10)

### Cómo se forman las clases

La tasa es una variable continua y se agrupa en cinco clases. Se comparan seis métodos que producen cinco grupos mediante la desviación absoluta alrededor de la mediana de cada clase (ADCM). Un ADCM menor indica que los valores reunidos dentro de cada clase son más semejantes.

A diferencia de seleccionar un método por costumbre, el código elige el menor ADCM obtenido con estos datos.

In [ ]:
np.random.seed(12345)

K = 5
the_variable = colombia_9377["tasa_homicidios_100000"]

classifiers = [
    mapclassify.EqualInterval(the_variable, k=K),
    mapclassify.Quantiles(the_variable, k=K),
    mapclassify.MaximumBreaks(the_variable, k=K),
    mapclassify.FisherJenks(the_variable, k=K),
    mapclassify.JenksCaspall(the_variable, k=K),
    mapclassify.MaxP(the_variable, k=K)
]

adcm_comparison = pd.DataFrame({
    "Clasificador": [classifier.name for classifier in classifiers],
    "ADCM": [classifier.adcm for classifier in classifiers]
}).sort_values("ADCM").reset_index(drop=True)

adcm_comparison

In [ ]:
adcm_comparison.sort_values("ADCM", ascending=False).plot.barh(
    x="Clasificador",
    y="ADCM",
    legend=False,
    color="#756bb1",
    figsize=(8, 5)
)
plt.title("Comparación de métodos de clasificación")
plt.xlabel("ADCM (menor es mejor)")
plt.ylabel("")
plt.show()

In [ ]:
selected_classifier = min(
    classifiers,
    key=lambda classifier: classifier.adcm
)

selected_name = selected_classifier.name
print("Clasificador seleccionado:", selected_name)
print("ADCM:", round(selected_classifier.adcm, 4))

colombia_9377["clase_codigo"] = selected_classifier.yb

level_labels = {
    0: "Muy bajo",
    1: "Bajo",
    2: "Medio",
    3: "Alto",
    4: "Muy alto"
}
ordered_labels = [level_labels[i] for i in range(K)]

colombia_9377["clase_tasa"] = pd.Categorical(
    colombia_9377["clase_codigo"].replace(level_labels),
    categories=ordered_labels,
    ordered=True
)

class_limits = pd.DataFrame({
    "Clase": ordered_labels,
    "Límite_superior": np.round(selected_classifier.bins, 2)
})

class_limits

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))

colombia_9377.plot(
    column="clase_tasa",
    cmap="YlOrRd",
    categorical=True,
    edgecolor="grey",
    linewidth=0.5,
    legend=True,
    legend_kwds={
        "title": "Nivel relativo",
        "loc": "lower left"
    },
    ax=ax
)

ax.set_title(
    "Homicidios por cada 100 000 habitantes en 2025\n"
    f"Clasificación {selected_name} ({K} clases)"
)
ax.set_xlim(*x_limits)
ax.set_ylim(*y_limits)
ax.set_axis_off()
plt.show()

## Lectura conjunta

Los tres mapas no son versiones intercambiables. El DDM y el PSM resaltan los departamentos que concentran una mayor cantidad total de homicidios registrados. Por ello, una unidad muy poblada puede destacar aunque su intensidad relativa no sea la más alta.

La coropleta relaciona los registros con la población y permite comparar departamentos de tamaños distintos. Esto explica por qué un departamento puede tener pocos puntos o un círculo pequeño y, al mismo tiempo, aparecer en una categoría alta: el número absoluto es reducido, pero resulta elevado en relación con sus habitantes.

La interpretación final debe conservar dos límites. Primero, la variable utilizada combina homicidio intencional y homicidio por tránsito vehicular. Segundo, los resultados cubren los 32 departamentos y excluyen Bogotá D. C. porque su categoría administrativa es Distrito Capital. Por esa razón, las sumas del cuaderno no equivalen al total nacional que incluiría Bogotá.